# Day 11: NumPy 入门 —— 数组、向量化、布尔索引、axis（练习）

> **目标**: 掌握 NumPy 核心操作，理解「向量化」替代循环的思想，为 Pandas 和量化计算打地基。
> **环境**: `import numpy as np`

## 1. 数组创建 —— 从 Python list 到 ndarray

NumPy 的核心是 `ndarray`（N-dimensional array）。它要求所有元素**同类型**，这换来了极致的运算速度。

In [1]:
import numpy as np

# 从 list 创建
a = np.array([1, 2, 3, 4, 5])
print(a)          # [1 2 3 4 5]
print(type(a))    # <class 'numpy.ndarray'>
print(a.dtype)    # int64（元素类型）
print(a.shape)    # (5,) —— 一维，5个元素
print(a.ndim)     # 1 —— 维度数

[1 2 3 4 5]
<class 'numpy.ndarray'>
int64
(5,)
1


In [2]:
# 二维数组（矩阵）
b = np.array([[1, 2, 3],
              [4, 5, 6]])
print(b.shape)    # (2, 3) —— 2行3列
print(b.ndim)     # 2

(2, 3)
2


In [3]:
# 快捷创建函数
print(np.zeros((2, 3)))      # 全0，形状(2,3)
print(np.ones((3, 2)))       # 全1
print(np.arange(0, 10, 2))   # [0 2 4 6 8]，步长2
print(np.linspace(0, 1, 5))  # [0. 0.25 0.5 0.75 1.]，等分5个点
print(np.random.rand(2, 3))  # [0,1)均匀分布随机数

[[0. 0. 0.]
 [0. 0. 0.]]
[[1. 1.]
 [1. 1.]
 [1. 1.]]
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[[0.47546565 0.33363105 0.26179095]
 [0.1599992  0.53771882 0.0630984 ]]


## 2. 向量化运算 —— 告别 for 循环

**核心思想**: NumPy 数组支持「广播」运算，对整个数组做一次操作，比 Python for 循环快 10-100 倍。这是数据岗必须养成的习惯。

In [4]:
prices = np.array([100, 150, 200, 250])

# 向量化：数组 + 标量
print(prices * 1.1)          # 每个元素涨10%

# 向量化：数组 + 数组（对应元素）
quantities = np.array([10, 5, 8, 3])
print(prices * quantities)   # 每个订单的金额

# 向量化：数组 比较 标量 → 布尔数组
print(prices > 150)          # [False False  True  True]

[110. 165. 220. 275.]
[1000  750 1600  750]
[False False  True  True]


In [5]:
# 对比：循环 vs 向量化（体会速度差异）
import time

n = 1_000_000
python_list = list(range(n))
numpy_arr = np.arange(n)

# Python 循环
start = time.time()
result_loop = [x * 2 for x in python_list]
t_loop = time.time() - start

# NumPy 向量化
start = time.time()
result_vec = numpy_arr * 2
t_vec = time.time() - start

print(f"Python list: {t_loop:.4f}s")
print(f"NumPy array: {t_vec:.4f}s")
print(f"加速比: {t_loop / t_vec:.0f}x")

Python list: 0.0825s
NumPy array: 0.0026s
加速比: 32x


## 3. 布尔索引 —— 筛选数据的核心技巧

用布尔数组当「面具」套在数组上，只保留 True 对应位置的元素。这是 Pandas 筛选的行版本。

In [6]:
returns = np.array([0.05, -0.02, 0.08, -0.01, 0.12, -0.05])  # 日收益率

# 筛选正收益
positive = returns[returns > 0]
print(positive)   # [0.05 0.08 0.12]

# 筛选 + 同时改值（把负收益抹成0）
returns_clipped = returns.copy()
returns_clipped[returns_clipped < 0] = 0
print(returns_clipped)  # [0.05 0.    0.08 0.    0.12 0.   ]

# 组合条件：收益>0 且 收益<0.1
mid = returns[(returns > 0) & (returns < 0.1)]  # & 是「按位与」，对应元素都为True才保留
print(mid)        # [0.05 0.08]
# ⚠️ 注意：组合条件必须用 & 和括号，不能用 Python 的 and

[0.05 0.08 0.12]
[0.05 0.   0.08 0.   0.12 0.  ]
[0.05 0.08]


## 4. axis 参数 —— 降维的方向

`axis=0` = 沿着**行**方向压下去（对每列做操作，结果行数减少）
`axis=1` = 沿着**列**方向压过去（对每行做操作，结果列数减少）

**记忆口诀**: axis 指定的是「被消灭的维度」。axis=0 消灭行，留下列的汇总。

In [7]:
# 模拟 3只股票 × 4天的收盘价
closes = np.array([
    [100, 102, 101, 105],   # 股票A
    [ 50,  48,  52,  51],   # 股票B
    [200, 198, 202, 205]    # 股票C
])
print("形状:", closes.shape)  # (3, 4)

# axis=0：按列汇总（每只股票4天的平均）→ 结果长度=4（每天3只股票的平均）
print("每天平均收盘价:", closes.mean(axis=0))  # [116.67, 116., 118.33, 120.33]

# axis=1：按行汇总（每只股票4天的平均）→ 结果长度=3
print("每只股票平均:", closes.mean(axis=1))   # [102.   50.25 201.25]

# 其他常用聚合
print("每只股票最大:", closes.max(axis=1))
print("每天最小:", closes.min(axis=0))
print("每只股票波动（标准差）:", closes.std(axis=1))

形状: (3, 4)
每天平均收盘价: [116.66666667 116.         118.33333333 120.33333333]
每只股票平均: [102.    50.25 201.25]
每只股票最大: [105  52 205]
每天最小: [50 48 52 51]
每只股票波动（标准差）: [1.87082869 1.47901995 2.58602011]


## 5. 切片与 reshape —— 数组变形

NumPy 切片和 Python list 类似，但支持多维切片。

In [8]:
arr = np.arange(12).reshape(3, 4)  # 0~11 排成 3行4列
print(arr)
# [[ 0  1  2  3]
#  [ 4  5  6  7]
#  [ 8  9 10 11]]

print(arr[0, :])      # 第0行 [0 1 2 3]
print(arr[:, 1])      # 第1列 [1 5 9]
print(arr[1:, 2:])    # 第1行起，第2列起 [[ 6  7]
                      #                             [10 11]]

# reshape：改变形状，元素总数不变
print(arr.reshape(2, 6))   # 2行6列
print(arr.reshape(-1, 3))  # -1 表示「自动算」，变成 ?行3列

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[0 1 2 3]
[1 5 9]
[[ 6  7]
 [10 11]]
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]


## 6. 常用数学函数 —— 数据岗高频

这些函数在量化分析中会反复用到。

In [9]:
data = np.array([10, 20, 30, 40, 50])

print(np.sum(data))       # 150
print(np.mean(data))      # 30.0
print(np.std(data))       # 标准差（总体）
print(np.var(data))       # 方差
print(np.max(data))       # 50
print(np.min(data))       # 10
print(np.argmax(data))    # 最大值的索引：4
print(np.argmin(data))    # 最小值的索引：0

# 累计和 / 累计积
print(np.cumsum(data))    # [ 10  30  60 100 150]
print(np.cumprod(data))   # [   10   200  6000 240000 12000000]

# 百分位数
print(np.percentile(data, 50))  # 中位数：30.0
print(np.median(data))          # 中位数

150
30.0
14.142135623730951
200.0
50
10
4
0
[ 10  30  60 100 150]
[      10      200     6000   240000 12000000]
30.0
30.0


## 7. 广播（Broadcasting）初识

当两个数组形状不同时，NumPy 会尝试「广播」它们，让形状兼容。这是向量化的高级形态。

In [10]:
# 例：矩阵 + 向量（每列加一个不同的数）
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])   # shape (2, 3)
bias = np.array([10, 20, 30])    # shape (3,)

# bias 被「拉伸」成 [[10,20,30], [10,20,30]]，然后对应相加
print(matrix + bias)
# [[11 22 33]
#  [14 25 36]]

# 例：每行归一化（每行除以该行的和）
row_sums = matrix.sum(axis=1, keepdims=True)  # keepdims=True 保持维度，shape (2,1)
print("行和:", row_sums)
print("归一化:", matrix / row_sums)  # (2,3) / (2,1) → 广播

[[11 22 33]
 [14 25 36]]
行和: [[ 6]
 [15]]
归一化: [[0.16666667 0.33333333 0.5       ]
 [0.26666667 0.33333333 0.4       ]]


## 8. 数据类型转换 —— 量化场景常见

从 CSV/数据库读来的数据默认是字符串，需要显式转换。

In [11]:
# 创建时指定类型
a = np.array([1, 2, 3], dtype=np.float64)  # 强制浮点
print(a.dtype)

# 类型转换
b = np.array(["1.5", "2.5", "3.5"])
b_float = b.astype(float)
print(b_float)        # [1.5 2.5 3.5]
print(b_float.dtype)  # float64

# 布尔转整数（True=1, False=0）
mask = np.array([True, False, True])
print(mask.astype(int))  # [1 0 1]

float64
[1.5 2.5 3.5]
float64
[1 0 1]


## 今日要点总结

| 概念 | 要点 |
|------|------|
| `ndarray` | 同类型、多维、向量化运算 |
| 创建 | `array/zeros/ones/arange/linspace/random.rand` |
| 向量化 | 数组 ±×÷ 标量/数组，不用 for 循环 |
| 布尔索引 | `arr[arr > 0]`，组合条件用 `&` + 括号 |
| axis | `axis=0` 压行（列汇总），`axis=1` 压列（行汇总） |
| 广播 | 形状不同的数组自动对齐，实现高级向量化 |
| 聚合 | `sum/mean/std/max/min/argmax/percentile` |

**核心心法**: 看到循环操作数组，先想「能不能向量化」。